In [0]:
   %sql
   CREATE VOLUME IF NOT EXISTS fmcg.gold.parent_incremental_data;

In [0]:
%sql
COPY INTO fmcg.gold.fact_orders
FROM (
  SELECT CAST(date AS DATE) AS date, product_code, customer_code, CAST(sold_quantity AS BIGINT) AS sold_quantity
  FROM '/Volumes/fmcg/gold/parent_incremental_data/fact_orders.csv'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true');

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
4485,4485,0


In [0]:
%sql
SELECT COUNT(*) AS parent_dec_rows
FROM fmcg.gold.fact_orders f
WHERE f.date = '2025-12-01'
  AND NOT EXISTS (SELECT 1 FROM fmcg.silver.products p WHERE p.product_code = f.product_code);

parent_dec_rows
4485


In [0]:
%sql
SELECT COUNT(*) AS parent_rows,
       SUM(sold_quantity) AS parent_qty,
       MIN(date) AS first_month,
       MAX(date) AS last_month
FROM fmcg.gold.fact_orders f
WHERE NOT EXISTS (SELECT 1 FROM fmcg.silver.products p WHERE p.product_code = f.product_code);

parent_rows,parent_qty,first_month,last_month
97540,27836359,2024-01-01,2025-12-01
